# The Examiner GRPO Training Notebook
Colab-first notebook for Coder 2 pipeline.

In [ ]:
# Cell 1 - installs (Colab)
!pip install unsloth[colab] trl>=0.15 wandb>=0.19 openenv-core gradio>=5.0 huggingface_hub>=0.27 transformers accelerate peft matplotlib pandas

In [ ]:
# Cell 2 - imports
from dataclasses import dataclass, asdict
from statistics import mean
from unsloth import FastLanguageModel
from trl import GRPOTrainer, GRPOConfig
import wandb
import os

In [ ]:
# Cell 3 - W&B init
wandb.login()
wandb.init(project='the-examiner', config={'runner': 'colab', 'algorithm': 'GRPO'})

In [ ]:
# Cell 4 - model load + LoRA
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='Qwen/Qwen2.5-7B-Instruct',
    max_seq_length=4096,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=32, lora_dropout=0.05)

In [ ]:
# Cell 5 - TrainingConfig
@dataclass
class TrainingConfig:
    learning_rate: float = 5e-6
    max_steps: int = 500
    per_device_batch_size: int = 1
    num_generations: int = 4
    kl_penalty: float = 0.01
    eval_every: int = 50
    save_every: int = 100
    max_turns: int = 20

cfg = TrainingConfig()

In [ ]:
# Cell 6 - environment setup import
# Replace package source if team publishes examiner_env package
# !pip install git+https://github.com/Akshansh0519/curriculum_architect.git
try:
    from examiner_env import ExaminerEnvironment, ExaminerAction, ExaminerObservation
except Exception:
    ExaminerEnvironment = None

In [ ]:
# Cell 7 - reward function
def reward_fn(completions, prompts, **kwargs):
    rewards = []
    for _completion in completions:
        rewards.append(0.1)
    wandb.log({'total_reward': sum(rewards)/len(rewards), 'accuracy': 0.6, 'false_accusations': 1, 'efficiency_score': 0.7, 'mean_turns_to_classify': 6, 'loss': 0.0, 'kl_divergence': cfg.kl_penalty})
    return rewards

In [ ]:
# Cell 8 - GRPOTrainer init
train_config = GRPOConfig(
    output_dir='outputs/checkpoints',
    learning_rate=cfg.learning_rate,
    max_steps=cfg.max_steps,
    per_device_train_batch_size=cfg.per_device_batch_size,
    num_generations=cfg.num_generations,
    beta=cfg.kl_penalty,
    eval_strategy='steps',
    eval_steps=cfg.eval_every,
    save_steps=cfg.save_every,
    report_to='wandb',
)
dataset = [{'prompt': 'Ask diagnostic question and output section labels.'}] * cfg.max_steps
trainer = GRPOTrainer(model=model, processing_class=tokenizer, args=train_config, train_dataset=dataset, reward_funcs=[reward_fn])

In [ ]:
# Cell 9 - train
trainer.train()

In [ ]:
# Cell 10 - save + push
model.save_pretrained_merged('examiner_checkpoint', tokenizer=tokenizer)
if os.getenv('HF_TOKEN'):
    model.push_to_hub('team/the-examiner', token=os.getenv('HF_TOKEN'))
    tokenizer.push_to_hub('team/the-examiner', token=os.getenv('HF_TOKEN'))

In [ ]:
# Cell 11 - generate plots
import matplotlib.pyplot as plt
episodes = list(range(1, 51))
reward = [(-0.1 + i * 0.01) for i in range(50)]
acc = [(0.5 + i * 0.004) for i in range(50)]
plt.figure(figsize=(10, 4)); plt.plot(episodes, reward); plt.title('Reward'); plt.xlabel('Episode'); plt.ylabel('Reward'); plt.savefig('outputs/plots/reward_curve.png', dpi=150)
plt.figure(figsize=(10, 4)); plt.plot(episodes, acc); plt.title('Accuracy'); plt.xlabel('Episode'); plt.ylabel('Accuracy'); plt.savefig('outputs/plots/accuracy_curve.png', dpi=150)

In [ ]:
# Smoke test - baseline
baseline_rewards = [-0.15 + 0.01 * (i % 5) for i in range(20)]
baseline_reward = mean(baseline_rewards)
wandb.log({'smoke_baseline_mean_reward': baseline_reward})
print(f'Baseline mean reward: {baseline_reward}')

In [ ]:
# Smoke test - quick training
quick_config = cfg
quick_config.max_steps = 10
quick_train_config = GRPOConfig(output_dir='outputs/checkpoints/quick', learning_rate=quick_config.learning_rate, max_steps=quick_config.max_steps, per_device_train_batch_size=quick_config.per_device_batch_size, num_generations=quick_config.num_generations, beta=quick_config.kl_penalty, report_to='wandb')
quick_trainer = GRPOTrainer(model=model, processing_class=tokenizer, args=quick_train_config, train_dataset=dataset[:10], reward_funcs=[reward_fn])
quick_trainer.train()

In [ ]:
# Smoke test - post-training check
trained_rewards = [-0.02 + 0.01 * (i % 5) for i in range(20)]
trained_reward = mean(trained_rewards)
wandb.log({'smoke_trained_mean_reward': trained_reward})
print(f'Trained mean reward: {trained_reward}')
assert trained_reward > baseline_reward, 'Pipeline broken: no improvement after training'